In [10]:
import openai
from qdrant_client import QdrantClient

from langsmith import Client
from qdrant_client import QdrantClient

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
import os
from superlinked import framework as sl
from api.agents.superlinked_app.index import business_index, business
from api.agents.superlinked_app.query import query
from api.agents.superlinked_app.utils.utils import *

### Download an example reference data point from LangSmith

In [2]:
client = Client()

In [3]:
dataset = client.read_dataset(
    dataset_name="yelp-rag-evaluation-dataset"
)

In [4]:
dataset

Dataset(name='yelp-rag-evaluation-dataset', description='Dataset for evaluating RAG pipeline', data_type=<DataType.kv: 'kv'>, id=UUID('071ace9b-36a7-457d-825c-4aa735ea2856'), created_at=datetime.datetime(2026, 4, 5, 5, 46, 10, 332558, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 4, 5, 5, 46, 10, 332558, tzinfo=TzInfo(0)), example_count=117, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-26.3.1-arm64-arm-64bit', 'sdk_version': '0.7.23', 'runtime_version': '3.12.12', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [5]:
list(client.list_examples(dataset_id=dataset.id, limit=10))[1].outputs

{'ground_truth': "Sabrosito's (5011 W Hillsborough Ave) is categorized as Cuban and lists RestaurantsDelivery = True, so it offers delivery for Cuban cuisine.",
 'reference_context_ids': ['GVNehsYuSB-fFX0FQAuI-Q'],
 'reference_descriptions': ["Sabrosito's"]}

In [6]:
list(client.list_examples(dataset_id=dataset.id, limit=10))[1].inputs

{'question': "Which restaurants serve 'Cuban' cuisine and also offer delivery?"}

In [7]:
reference_input = list(client.list_examples(dataset_id=dataset.id, limit=10))[1].inputs
reference_output = list(client.list_examples(dataset_id=dataset.id, limit=10))[1].outputs

### RAG Pipeline

In [11]:
qdrant_vdb = sl.QdrantVectorDatabase(
    url="http://localhost:6333",
    # Superlinked's QdrantVectorDatabase currently requires an api_key arg.
    # For local Qdrant this is typically unused, so we default to empty.
    api_key=os.getenv("QDRANT_API_KEY", ""),
)
parser = sl.DataFrameParser(business)

source_qdrant = sl.RestSource(
    business,
    parser=parser,
)

# RestExecutor needs sl.RestQuery (path for /api/v1/search/<query_path> by default).
business_rest_query = sl.RestQuery(
    rest_descriptor=sl.RestDescriptor(query_path="business_search"),
    query_descriptor=query,
)

executor_qdrant = sl.RestExecutor(
    sources=[source_qdrant],
    indices=[business_index],
    vector_database=qdrant_vdb,
    queries=[business_rest_query],
)
qdrant_app = executor_qdrant.run()


def Retrieve_context(question, qdrant_app, k=5):
    qdrant_results = qdrant_app.query(
    query,
    natural_query=question,
    limit=k,
)

    format_minute_columns_to_hhmm(sl.PandasConverter.to_pandas(qdrant_results))
    return qdrant_results


def _result_to_restaurants(result) -> list[dict[str, Any]]:
    df_columns = ["business_id", "name", "address", "city", "state", "postal_code", "latitude", "longitude", "stars", "review_count", "is_open", "categories", "attributes", "hours"]
    df = sl.PandasConverter.to_pandas(result).rename(columns={"id": "business_id"})
    # Derive `is_open` robustly.
    # Some payloads may contain only `is_open_i` (0/1) instead of the boolean `is_open`.
    if "is_open" not in df.columns:
        if "is_open_i" in df.columns:
            df["is_open"] = df["is_open_i"].astype(int)
        else:
            df["is_open"] = 0

    df = df.assign(
        categories=df.get("category_tags", df.get("categories_text")),
        is_open=df["is_open"].astype(int),
    )

    # Parse attributes/hours when present.
    for c in ("attributes", "hours"):
        if c in df.columns:
            df[c] = df[c].map(
                lambda v: json.loads(v) if isinstance(v, str) and v.strip()
                else ({} if v in ("", None) else v)
            )
        else:
            df[c] = {}
    return df.reindex(columns=df_columns).to_dict(orient="records")


def build_prompt(preprocessed_context, question):
    prompt=f"""
    You are a yelp shopping assistant that can answer question about the restaurants.
    You will be given a question and a list of context.
    Instructions:
    - You need to answer questions based on the provided context only
    - Never use the word context and rfer to it as the available businesses or amenities
    - respond naturally and provide as much details as possible to the user request 
    - Refrain from using filter sush as is_open =True ...Rather say open today

    Context:
    {preprocessed_context}

    Question:
    {question}
    """

    return prompt


def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role":"system", "content": prompt}],
        reasoning_effort="medium"
    )

    return response.choices[0].message.content


def rag_pipeline(question):
    context=Retrieve_context(question, qdrant_app)
    preprocessed_context=_result_to_restaurants(context)
    prompt=build_prompt(preprocessed_context, question)
    answer=generate_answer(prompt)

    return {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": [e.id for e in context.entries],
        "retrieved_restaurant_names": [e.fields.get("name") for e in context.entries],
        "similarity_score": [e.metadata.score for e in context.entries],
    }

/Users/wilfriedtcheumaha/Code/yelp-assistant/.venv/lib/python3.12/site-packages/superlinked/framework/storage/qdrant/qdrant_vdb_connector.py:87: UserWarning: Api key is used with an insecure connection.
  self._client = AsyncQdrantClient(
/Users/wilfriedtcheumaha/Code/yelp-assistant/.venv/lib/python3.12/site-packages/superlinked/framework/storage/qdrant/qdrant_vdb_connector.py:92: UserWarning: Api key is used with an insecure connection.
  self._sync_client = QdrantClient(


02:29:33 superlinked.framework.query.query_dag_evaluator INFO   initialized query dag
02:29:33 superlinked.framework.online.online_dag_evaluator INFO   initialized entity dag


In [12]:
rag_pipeline("looking for open cocktail bars in New Orleans that has happy hour with outdoor seating and garage parking")

02:30:39 superlinked.framework.common.delayed_evaluator INFO   Processed sentence-transformers/all-MiniLM-L6-v2 embed
02:30:39 superlinked.framework.query.query_dag_evaluator INFO   evaluated query
02:30:39 superlinked.framework.dsl.executor.query.query_executor INFO   executed query


{'answer': 'Trenasse in New Orleans fits all your criteria and is open today.\n\nDetails:\n- Where: Trenasse, 444 St Charles Ave, Ste 100, Intercontinental Hotel, New Orleans, LA 70130\n- Open now / open today: Yes (hours typically 11:00–22:00 daily)\n- Type: Cocktail bar (also listed under American and other categories)\n- Happy hour: Yes\n- Outdoor seating: Yes\n- Parking: Garage parking available (also street parking and valet; no validated parking)\n- Alcohol: Full bar\n- Price range: Moderate\n- Reservations: Available\n- Tableservice / Takeout / Delivery: Table service; Takeout available; Delivery not listed\n- Other handy notes: Casual attire; good for groups; wheelchair accessible; dogs allowed; free WiFi; average noise level\n\nIf you’d like, I can check today’s specific happy hour times or help you book a reservation. Would you like me to do that?',
 'question': 'looking for open cocktail bars in New Orleans that has happy hour with outdoor seating and garage parking',
 'retr

### RAG Metrics


In [13]:
from ragas.dataset_schema import SingleTurnSample 
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

/var/folders/rt/bmzmdtl92k7bz0smp4yx2f7r0000gn/T/ipykernel_27918/1973645927.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/rt/bmzmdtl92k7bz0smp4yx2f7r0000gn/T/ipykernel_27918/1973645927.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/rt/bmzmdtl92k7bz0smp4yx2f7r0000gn/T/ipykernel_27918/1973645927.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is depre

In [39]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

/var/folders/rt/bmzmdtl92k7bz0smp4yx2f7r0000gn/T/ipykernel_27918/2270190839.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
/var/folders/rt/bmzmdtl92k7bz0smp4yx2f7r0000gn/T/ipykernel_27918/2270190839.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [16]:
reference_input

{'question': "Which restaurants serve 'Cuban' cuisine and also offer delivery?"}

In [17]:
reference_output

{'ground_truth': "Sabrosito's (5011 W Hillsborough Ave) is categorized as Cuban and lists RestaurantsDelivery = True, so it offers delivery for Cuban cuisine.",
 'reference_context_ids': ['GVNehsYuSB-fFX0FQAuI-Q'],
 'reference_descriptions': ["Sabrosito's"]}

In [26]:
result = rag_pipeline(reference_input["question"])

02:39:11 superlinked.framework.query.query_dag_evaluator INFO   evaluated query
02:39:11 superlinked.framework.dsl.executor.query.query_executor INFO   executed query


In [27]:
result

{'answer': "Here are the Cuban restaurants that also offer delivery:\n\n- Havana Dreamers Cafe — 3104 Town Ave, Ste 107, Trinity, FL 34655. Cuban cuisine with delivery available; open today.\n- Box Of Cubans — 10323 Causeway Blvd, Tampa, FL 33619. Cuban cuisine with delivery available; open today.\n- Felicitous — 11706 N 51st St, Tampa, FL 33617. Cuban cuisine with delivery available; open today.\n- Sabrosito's — 5011 W Hillsborough Ave, Ste A, Tampa, FL 33634. Cuban cuisine with delivery available; open today.\n- Congreso Cubano — New Orleans, LA 70117. Cuban cuisine with delivery available; open today.\n\nIf you’d like, I can share quick details like hours for today or how far each place is from you.",
 'question': "Which restaurants serve 'Cuban' cuisine and also offer delivery?",
 'retrieved_context_ids': ['Gm2t6tXjy0mNudBFsBxeJA',
  'xYDO35xImuSEGO5fKk6UtA',
  'HqV4cganalEtEtKGALbzOQ',
  'GVNehsYuSB-fFX0FQAuI-Q',
  'l5jIYx7UQMfyX0Fc4HNCvA'],
 'retrieved_restaurant_names': ['Havana

In [32]:
async def ragas_faithfulness(run, example):

    sample = SingleTurnSample(
            user_input=run["question"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_restaurant_names"]
        )
    scorer = Faithfulness(llm=ragas_llm)

    return await scorer.single_turn_ascore(sample)

In [33]:
await ragas_faithfulness(result, "")

0.0

In [ ]:
async def ragas_response_relevancy(run, example):

    sample = SingleTurnSample(
            user_input=run["question"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_restaurant_names"]
        )
    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_response_relevancy(result, "")

np.float64(0.9300572723353273)

In [42]:
async def ragas_context_precision_id_based(run, example):

    sample = SingleTurnSample(
            retrieved_context_ids=run["retrieved_context_ids"],
            reference_context_ids=example["reference_context_ids"]
        )
    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [43]:
await ragas_context_precision_id_based(result, reference_output)

0.2

In [44]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
            retrieved_context_ids=run["retrieved_context_ids"],
            reference_context_ids=example["reference_context_ids"]
        )
    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [45]:
await ragas_context_recall_id_based(result, reference_output)

1.0